In [1]:
import numpy as np
import os
import time

import rosbag
import yaml

import meshcat
import meshcat.geometry as g
import meshcat.transformations as tf

from tempfile import TemporaryDirectory

from PIL import Image
import io
import rospy 
from math_utils import rotation_matrix_to_quaternion, transform_bundletrack_output
# video resolution
video_resolution=[640, 480]

#
## parameters:
#
cam = 'cam0'
video_name = 'old_toss_5'
odom_bag_file = './odom_10.bag'
cam_bag_file = './raw_10.bag'
cam_poses_file = './assets/realsense_pose_old.yaml'
output_file = './' + video_name + '.mp4'
###############OLD DATASET ###############
# start time
# start_time = rospy.rostime.Time(secs=1655404893, nsecs=899137)  # toss 1
# start_time = rospy.rostime.Time(secs=1655404906, nsecs=156495)  # toss 2
# start_time = rospy.rostime.Time(secs=1655404918, nsecs=435741)  # toss 3
# start_time = rospy.rostime.Time(secs=1655404930, nsecs=306306)  # toss 4
# start_time = rospy.rostime.Time(secs=1655404944, nsecs=447914)  # toss 5
# start_time=rospy.rostime.Time(secs=1655404955, nsecs=272877)  # toss 6
# start_time=rospy.rostime.Time(secs=1655404966, nsecs=698458)  # toss 7
# start_time=rospy.rostime.Time(secs=1655404977, nsecs=941630)  # toss 8
# start_time=rospy.rostime.Time(secs=1655404991, nsecs=641514) # toss 9
# start_time=rospy.rostime.Time(secs=1655405008, nsecs=720921) # toss 10

# end time
# end_time = rospy.rostime.Time(secs=1655404906, nsecs=156495)  # toss 1
# end_time = rospy.rostime.Time(secs=1655404918, nsecs=435741)  # toss 2
# end_time = rospy.rostime.Time(secs=1655404930, nsecs=306306)  # toss 3
# end_time = rospy.rostime.Time(secs=1655404944, nsecs=447914)  # toss 4
# end_time = rospy.rostime.Time(secs=1655404955, nsecs=272877)  # toss 5
# end_time=rospy.rostime.Time(secs=1655404966, nsecs=698458)  # toss 6
# end_time=rospy.rostime.Time(secs=1655404977, nsecs=941630)  # toss 7
# end_time=rospy.rostime.Time(secs=1655404991, nsecs=641514)  # toss 8
# end_time=rospy.rostime.Time(secs=1655405008, nsecs=720921)  # toss 9
# end_time=rospy.rostime.Time(secs=1655405022, nsecs=942549)  # toss 10
###

# <------------------------------- Redo toss separation 1/31/23 --------------------------------
# start time
# start_time = rospy.rostime.Time(secs=1655404893, nsecs=899137)  # toss 1
# start_time = rospy.rostime.Time(secs=1655404908, nsecs=279948)  # toss 2
# start_time = rospy.rostime.Time(secs=1655404920, nsecs=470680) # toss 3
# start_time = rospy.rostime.Time(secs=1655404932, nsecs=647236) # toss 4
start_time = rospy.rostime.Time(secs=1655404945, nsecs=387903)  # toss 5

# end time
# end_time = rospy.rostime.Time(secs=1655404908, nsecs=279948)  # toss 1
# end_time = rospy.rostime.Time(secs=1655404920, nsecs=470680)  # toss 2
# end_time = rospy.rostime.Time(secs=1655404932, nsecs=647236) # toss 3
# end_time = rospy.rostime.Time(secs=1655404945, nsecs=387903)  # toss 4
end_time = rospy.rostime.Time(secs=1655404955, nsecs=463919)  # toss 5

cam_topic = '/camera/color/image_raw'

intrinsics = [380.2484436035156, 379.8265380859375,314.2138977050781, 240.59800720214844,] # fx, fy, cx, cy

with open(cam_poses_file, 'r') as stream:
    data_loaded = yaml.safe_load(stream)
print(data_loaded[cam]['pose']['position']['x'])

cam_pos_dict = data_loaded[cam]['pose']['position']
cam_position = [cam_pos_dict['x'], cam_pos_dict['y'], cam_pos_dict['z']]
cam_rot_dict = data_loaded[cam]['pose']['rotation']
cam_orientation = [cam_rot_dict['x'], cam_rot_dict['y'], cam_rot_dict['z']]


# Compute T_WC, transform from world to camera
cam_angle = np.linalg.norm(cam_orientation)
cam_axis = cam_orientation/cam_angle
T_WC = tf.translation_matrix(cam_position) @ tf.quaternion_matrix(tf.quaternion_about_axis(cam_angle, cam_axis))
print(tf.quaternion_matrix(tf.quaternion_about_axis(cam_angle, cam_axis)))

# Given fov_x, fov_y, desire output to be h x w resolution
# set fov=fov_y, screen_h = h
# set screen_w = fov_x/fov_y * w

fx = intrinsics[0]
fy = intrinsics[1]

fov_x = 2*np.arctan(video_resolution[1]/(2*fy))*180/np.pi
fov_y = 2*np.arctan(video_resolution[0]/(2*fx))*180/np.pi


cam_fov = fov_x

print('Extracted field of view: ' + f'{cam_fov:f}')


resolution = video_resolution

vis = meshcat.Visualizer()

1.1416436
[[-0.09773456  0.48134989 -0.87106271  0.        ]
 [ 0.98992557 -0.04307843 -0.13487633  0.        ]
 [-0.10244672 -0.87546932 -0.47229031  0.        ]
 [ 0.          0.          0.          1.        ]]
Extracted field of view: 64.574913
You can open the visualizer by visiting the following URL:
http://127.0.0.1:7000/static/


In [2]:
# Create the cubes. Set the opacity of the "real" to > 0 if you want to see it, for invisible
vis["real_1"].set_object(g.Box([0.1048, 0.1048, 0.1048]),
                       g.MeshLambertMaterial(
                             color=0x00ff00,
                             reflectivity=0.0,
                             transparent=0,
                             opacity=.4))

# vis["real_1"].set_object(g.Box([0.096, 0.061, 0.096]),
#                        g.MeshLambertMaterial(
#                              color=0x00ff00,
#                              reflectivity=0.0,
#                              transparent=0,
#                              opacity=.4))

vis["real_2"].set_object(g.Box([0.1048, 0.1048, 0.1048]),
                       g.MeshLambertMaterial(
                             color=0xff22dd,
                             reflectivity=0.0,
                             transparent=0,
                             opacity=.4))


In [3]:
bag = rosbag.Bag(odom_bag_file)

# get summary info from rosbag as a dictionary
info = yaml.load(bag._get_yaml_info(), Loader=yaml.FullLoader)
TOPIC_STRING_1 = '/tagslam/odom/body_cube'
# TOPIC_STRING_2 = '/tagslam/odom/body_elbow_2'

# extract metadata from cube and board topics
elbow_1_topic = [topic for topic in info['topics'] if topic['topic'] == TOPIC_STRING_1][0]
# elbow_2_topic = [topic for topic in info['topics'] if topic['topic'] == TOPIC_STRING_2][0]

num_msg = elbow_1_topic['messages']

def extract_times(messages):
    t_ros = np.zeros(len(messages))
    for i, data in enumerate(list(messages)):
        (_, msg, _) = data
        tstamp = msg.header.stamp
        t_ros[i] = tstamp.secs + tstamp.nsecs * 1e-9

    return t_ros


def extract_poses(messages, start_time, end_time):
    # poses = np.zeros((7, len(messages)))
    poses = np.zeros((7,1))
    for i, data in enumerate(messages):
        (_, msg, _) = data
        if start_time <= msg.header.stamp < end_time:
            pose = msg.pose.pose
            pose_pos = np.asarray([pose.position.x, pose.position.y, pose.position.z])
            pose_quat = np.asarray([pose.orientation.w, pose.orientation.x, pose.orientation.y,
                                    pose.orientation.z])
            # print(np.hstack((pose_quat, pose_pos)).T.shape)
            poses = np.hstack((poses, np.hstack((pose_quat, pose_pos)).T.reshape((-1,1))))
            # poses[:4, i] = pose_quat
            # poses[4:7, i] = pose_pos
    return poses[:,1:]

def get_bundletrack_results():
    """
    State vector is 4 quaternion + 3 xyz position + 3 angular velocity + 3 linear velocity.
    """
    frame_num = len([name for name in os.listdir(DATA_DIR)])
    print("%i frames in total!"%frame_num)
    poses = np.zeros((7, frame_num))
    for frame_id in range(1, frame_num+1):
        pose = np.loadtxt(DATA_DIR + "%04i.txt" % frame_id)#in camera frame
        # For camera extrinsics
        CAMERA_CONFIG = {
            "old": {
                "translation": np.array([[1.14164360], [0.15815239], [0.66422200]]),
                "axis_vec": np.array([-1.57165949, -1.63112887, 1.07928078]),
            },
            "new": {
                "translation": np.array([[1.11076422], [-0.07966290], [0.67947702]]),
                "axis_vec": np.array([-1.61997882, -1.56988553, 0.86362178]),
            },
        }
        pose = transform_bundletrack_output(
            pose,
            DATA_DIR,
            ODOM_FILE_PATH,
        )
        pose_quat = rotation_matrix_to_quaternion(pose)
        pose_quat = pose_quat.reshape(1, -1)
        pose_pos = np.array(pose[:3, 3])
        poses[:4, frame_id-1] = pose_quat
        poses[4:7, frame_id-1] = pose_pos
    return poses

DATA_DIR = "/home/cnets-vision/mengti_ws/BundleSDF/results/old_toss_5/ob_in_cam/"
ODOM_FILE_PATH = ("/home/cnets-vision/mengti_ws/BundleSDF/data/old_toss_5/annotated_poses/")

bundletrack_poses = get_bundletrack_results()
# t_ros = extract_times(list(bag.read_messages(topics=[TOPIC_STRING_1])))
elbow_1 = extract_poses(list(bag.read_messages(topics=[TOPIC_STRING_1])), start_time, end_time)
# t_ros_2 = extract_times(list(bag.read_messages(topics=[TOPIC_STRING_2])))
# elbow_2 = extract_poses(list(bag.read_messages(topics=[TOPIC_STRING_2])))
print("elbow1: ", elbow_1.shape)
if (bundletrack_poses.shape[1]>elbow_1.shape[1]):
    bundletrack_poses = bundletrack_poses[:, :elbow_1.shape[1]]
print("bundletrack: ", bundletrack_poses.shape)
bag.close()

214 frames in total!
elbow1:  (7, 300)
bundletrack:  (7, 214)


In [5]:
from cv_bridge import CvBridge

# Load video
raw_bag = rosbag.Bag(cam_bag_file)
cam_messages= list(raw_bag.read_messages(topics=[cam_topic]))

num_cam_msg = len(cam_messages)
# print(num_cam_msg)
# extract camera
t_cam = np.zeros(len(cam_messages))
cam_data = []
bridge = CvBridge()
for i, data in enumerate(cam_messages):
    (_, msg, _) = data
    tstamp = msg.header.stamp
    if start_time <= tstamp < end_time:
        t_cam[i] = tstamp.secs + tstamp.nsecs * 1e-9
        # cam_data.append(msg.data)
        cv_img = bridge.imgmsg_to_cv2(msg, desired_encoding="passthrough")
        cam_data.append(cv_img)
print(len(cam_data))

300


In [6]:
base_url = "http://127.0.0.1"

meshcat_url = base_url + ":" + vis.url().split(":")[-1]

''' Create precisely-sized iframe with meshcat view; put this in its own Jupyter cell. '''
from IPython.display import HTML
frame_html = """
<div style="height: {height}px; width: {width}px; overflow-x: visible; overflow-y: visible; resize: none">
    <iframe src="{url}" style="width: 100%; height: 100%; border: none"></iframe>
</div>
""".format(url=meshcat_url, width=resolution[0], height=resolution[1])
HTML(frame_html)

In [8]:
# Frames
# (W) World
# (M) Meshcat
# (C) Camera
# (A) link_1
# (B) link_2

vis["cam"].set_transform(T_WC)
vis["cam_view"].set_transform(T_WC @ tf.translation_matrix([0,0,.05]))

# T_MC, look along z-axis but rotote by 180 degrees
T_MC = tf.translation_matrix([0, 0, -1]) @ tf.rotation_matrix(np.pi, (0,0,1))

T_MW = T_MC @ tf.inverse_matrix(T_WC)

cam = g.PerspectiveCamera(fov=cam_fov, zoom=1, aspect=640/480)
vis["/Cameras/default/rotated"].set_object(cam)

# vis["/Cameras/default/rotated/<object>"].set_property("zoom", 1)
vis["/Cameras/default/rotated/<object>"].set_property("position", [0,0,0])
vis["/Cameras/default"].set_transform(T_MC)



In [10]:
# view in meshcat save to images
# Turn off background, axes, and grid.
vis['/Background'].set_property("visible", False)
vis['/Grid'].set_property("visible", False)
vis['/Axes'].set_property("visible", False)

with TemporaryDirectory(prefix="ros-process-") as tmpdir:
    print(tmpdir)
    for i, pose in enumerate(bundletrack_poses.T):
        print('Processing frame ' + f'{i:d}' + ' of ' + f'{bundletrack_poses.shape[1]:d}', end='\r')
        print(i)
        # im = Image.open(io.BytesIO(cam_data[i])).convert('RGB')
        im = Image.fromarray(cam_data[i]).convert('RGB')
        T_WA = tf.translation_matrix(pose[4:7]) @ tf.quaternion_matrix(pose[:4])
        vis["real_1"].set_transform(T_MW @ T_WA)

        # pose_2 = elbow_1[:,i]
        # T_WB = tf.translation_matrix(pose_2[4:7]) @ tf.quaternion_matrix(pose_2[:4])
        # vis["real_2"].set_transform(T_MW @ T_WB)

        mesh_im = vis.get_image()
        # print(mesh_im.size)
        # mesh_im.show()
        im.paste(mesh_im, (0,0), mask = mesh_im)
        # im.show()
        # break
        im.save(tmpdir + '/' + f'{i:07d}' + '.png', format="png")
    os.system('ffmpeg -y -r 150 -i ' + tmpdir + '/%07d.png -vcodec libx264 -preset slow -crf 18 ' + output_file)

/tmp/ros-process-j8_42pmp
0rocessing frame 0 of 214
1rocessing frame 1 of 214
2rocessing frame 2 of 214
3rocessing frame 3 of 214
4rocessing frame 4 of 214
5rocessing frame 5 of 214
6rocessing frame 6 of 214
7rocessing frame 7 of 214
8rocessing frame 8 of 214
9rocessing frame 9 of 214
10ocessing frame 10 of 214
11ocessing frame 11 of 214
12ocessing frame 12 of 214
13ocessing frame 13 of 214
14ocessing frame 14 of 214
15ocessing frame 15 of 214
16ocessing frame 16 of 214
17ocessing frame 17 of 214
18ocessing frame 18 of 214
19ocessing frame 19 of 214
20ocessing frame 20 of 214
21ocessing frame 21 of 214
22ocessing frame 22 of 214
23ocessing frame 23 of 214
24ocessing frame 24 of 214
25ocessing frame 25 of 214
26ocessing frame 26 of 214
27ocessing frame 27 of 214
28ocessing frame 28 of 214
29ocessing frame 29 of 214
30ocessing frame 30 of 214
31ocessing frame 31 of 214
32ocessing frame 32 of 214
33ocessing frame 33 of 214
34ocessing frame 34 of 214
35ocessing frame 35 of 214
36ocessing f

ffmpeg version 4.2.7-0ubuntu0.1 Copyright (c) 2000-2022 the FFmpeg developers
  built with gcc 9 (Ubuntu 9.4.0-1ubuntu1~20.04.1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-avresample --disable-filter=resample --enable-avisynth --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librsvg --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --e